# Phase 3: Exploratory Data Analysis
This notebook uses the project loader and the discovered CSV schema. It does not train a model.

In [1]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
from src.data_loader import discover_csv, load_dataset, infer_target_column, summarize_dataset, encode_target
sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 100)

In [ ]:
DATA_PATH = discover_csv()
data = load_dataset(DATA_PATH)
target_column = infer_target_column(data)
target = encode_target(data[target_column])
summary = summarize_dataset(data, DATA_PATH, target_column)
print(f"Dataset: {summary['filename']} | rows={summary['rows']} | columns={summary['columns']}")
print(f"Target: {target_column}")
display(data.head())

## 1. Dataset overview and data types

In [ ]:
display(pd.DataFrame({'dtype': data.dtypes.astype(str), 'missing': data.isna().sum(), 'unique': data.nunique(dropna=False)}))
print(f"Duplicate rows: {data.duplicated().sum()}")

## 2. Missing values and duplicate analysis

In [ ]:
missing = data.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]
if missing.empty:
    print('No missing values found.')
else:
    display(missing.rename('missing_count').to_frame())
print(f"Duplicate rows: {data.duplicated().sum()}")

## 3. Fraud versus non-fraud distribution

In [ ]:
class_counts = target.map({0: 'Non-fraud', 1: 'Fraud'}).value_counts().reindex(['Non-fraud', 'Fraud'])
display(class_counts.rename('count').to_frame())
ax = class_counts.plot(kind='bar', color=['#2a9d8f', '#e76f51'], figsize=(6, 4), title='Target class distribution')
ax.set_xlabel('Class'); ax.set_ylabel('Transactions'); plt.xticks(rotation=0); plt.show()

## 4. Transaction amount and important numerical features

In [ ]:
numeric_columns = data.select_dtypes(include='number').columns.tolist()
numeric_features = [column for column in numeric_columns if column != target_column]
amount_candidates = [column for column in numeric_features if any(token in str(column).lower() for token in ('amount', 'amt', 'value'))]
amount_column = amount_candidates[0] if amount_candidates else None
if amount_column:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.histplot(data=data, x=amount_column, hue=target.map({0: 'Non-fraud', 1: 'Fraud'}), bins=40, element='step', stat='density', common_norm=False, ax=axes[0])
    sns.boxplot(data=data, x=target.map({0: 'Non-fraud', 1: 'Fraud'}), y=amount_column, ax=axes[1])
    axes[0].set_title(f'{amount_column} distribution'); axes[1].set_title(f'{amount_column} by class'); plt.tight_layout(); plt.show()
else:
    print('No numeric column with an amount-like name was found; no amount plot is assumed.')

plot_columns = numeric_features[:6]
if plot_columns:
    data[plot_columns].hist(bins=30, figsize=(14, 9), layout=(-1, 3))
    plt.suptitle('Numeric feature distributions'); plt.tight_layout(); plt.show()
else:
    print('No numeric features are available for distribution plots.')

## 5. Correlation and outlier analysis

In [ ]:
if numeric_features:
    correlation = data[numeric_features + [target_column]].corr(numeric_only=True)
    plt.figure(figsize=(max(7, len(correlation) * 0.7), max(5, len(correlation) * 0.55)))
    sns.heatmap(correlation, cmap='vlag', center=0, annot=len(correlation) <= 12, fmt='.2f')
    plt.title('Numeric feature and target correlations'); plt.tight_layout(); plt.show()

    outlier_rows = {}
    for column in numeric_features:
        values = data[column].dropna()
        q1, q3 = values.quantile([0.25, 0.75])
        iqr = q3 - q1
        outlier_rows[column] = int(((values < q1 - 1.5 * iqr) | (values > q3 + 1.5 * iqr)).sum())
    display(pd.Series(outlier_rows, name='iqr_outlier_count').sort_values(ascending=False).to_frame())
else:
    print('Correlation and IQR outlier analysis require numeric features.')

## 6. Initial observations and preprocessing requirements
Observations below are generated from the loaded data; no feature meaning is inferred beyond column names used for plotting.

- Confirm the target column and fraud rate from the class table above.
- Remove exact duplicate records before splitting, and report how many were removed.
- Impute missing numeric values with training-set medians and categorical values with the training-set mode.
- One-hot encode categorical features and scale numeric features inside a train-fitted pipeline.
- Preserve validation and test distributions with stratified splits.
- Treat IQR outliers as an investigation signal, not an automatic deletion rule.
- Compare class-weighted learning with sampling strategies only on training data; do not apply SMOTE blindly.

No ML model is trained in this notebook.